# LoRA persona DPO (TRL `DPOTrainer`, SmolLM2-360M-Instruct)
Trains a LoRA adapter on `data/persona_lora/ramsay_style_preferences.json`. No input dataset is needed.

## Setup
Set `GITHUB_USER` below. The repository is private, so add a Kaggle secret named `GITHUB_TOKEN` (a GitHub personal access token with read access) under *Add-ons → Secrets*. The token is passed as an HTTP header and is not written into the cloned repository.

In [ ]:
import base64
import os
import subprocess

from kaggle_secrets import UserSecretsClient

GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "Small-Model_Alignment"
REPO_DIR = f"/kaggle/working/{REPO_NAME}"

if not os.path.isdir(REPO_DIR):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    credentials = base64.b64encode(f"{GITHUB_USER}:{token}".encode()).decode()
    clone_command = [
        "git",
        "-c",
        f"http.extraHeader=Authorization: Basic {credentials}",
        "clone",
        "-q",
        f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git",
        REPO_DIR,
    ]
    result = subprocess.run(clone_command)
    if result.returncode != 0:
        raise SystemExit("git clone failed")

os.chdir(REPO_DIR)
print(os.getcwd())

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers==4.57.6" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib "trl==1.16.0"
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi

In [ ]:
import datasets
import peft
import torch
import transformers
import trl

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("trl", trl.__version__)
print("peft", peft.__version__)
print("visible GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
if torch.cuda.device_count() < 2:
    raise SystemExit("Expected two GPUs. Select the GPU T4 x2 accelerator.")

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py

## Train
`CUDA_VISIBLE_DEVICES=0` keeps the Trainer on one GPU, so the effective batch size stays 4 x 2 = 8.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/train_lora_persona.py --config configs/lora_persona.yaml --output_dir outputs/lora_persona

In [ ]:
!python scripts/plot_metrics.py --metrics_path outputs/lora_persona/metrics.jsonl --output_dir outputs/lora_persona/plots --run_name lora_persona

## Archive small files
Use *Save Version* afterwards so the `best/` and `final/` adapters are kept in the notebook output.

In [ ]:
!cd outputs && zip -q -r /kaggle/working/lora_persona_run_outputs.zip lora_persona/metrics.jsonl lora_persona/config_used.json lora_persona/train_log.txt lora_persona/plots
!ls -lh /kaggle/working/lora_persona_run_outputs.zip